## (Trained using Kaggle Notebook)

The format might be differ, but results and workflows are expected and configured beforehand

### TFT Conclusion

The baseline Temporal Fusion Transformer (TFT) was trained and evaluated across three chronological development folds. A preliminary three-trial Optuna search was conducted to explore alternative hyperparameter configurations. However, the best trial achieved a mean validation loss of 0.677433, approximately 0.43% higher than the baseline's 0.674543. Therefore, the baseline configuration was retained as the selected TFT configuration. No further tuning was justified by the observed results. The baseline model will proceed to internal-test evaluation before the final comparison with other forecasting models.

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
import lightning as L
import pytorch_forecasting as pf

print("PyTorch:", torch.__version__)
print("Lightning:", L.__version__)
print("PyTorch Forecasting:", pf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("\nGPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ))

TFT_INPUT_DIR = (
    "/kaggle/input/datasets/ryancjc/tft-salesf/tft"
)

print("\nInput directory exists:", os.path.exists(TFT_INPUT_DIR))

for root, dirs, files in os.walk(TFT_INPUT_DIR):
    level = root.replace(TFT_INPUT_DIR, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}  {file}")

PyTorch: 2.11.0+cu130
Lightning: 2.6.6
PyTorch Forecasting: 1.8.0
NumPy: 2.1.3
Pandas: 2.3.3

GPU available: True
GPU: Tesla T4
GPU memory (GB): 14.56

Input directory exists: True
tft/
  development_fold_1_training.pt
  development_fold_1_validation.pt
  development_fold_2_training.pt
  development_fold_2_validation.pt
  development_fold_3_training.pt
  development_fold_3_validation.pt
  internal_test_training.pt
  internal_test_validation.pt
  official_evaluation_training.pt
  official_evaluation_validation.pt
  tft_dataset_config.json
  preprocessors/
    development_fold_1_preprocessors.joblib
    development_fold_2_preprocessors.joblib
    development_fold_3_preprocessors.joblib
    internal_test_preprocessors.joblib
    official_evaluation_preprocessors.joblib


In [ ]:
import os
import json
import torch

CONFIG_PATH = os.path.join(TFT_INPUT_DIR, "tft_dataset_config.json")

with open(CONFIG_PATH, "r") as f:
    tft_config = json.load(f)

print("TFT configuration:")
print(json.dumps(tft_config, indent=2))

FOLD1_TRAIN_PATH = os.path.join(
    TFT_INPUT_DIR, "development_fold_1_training.pt"
)

FOLD1_VAL_PATH = os.path.join(
    TFT_INPUT_DIR, "development_fold_1_validation.pt"
)

train_dataset = torch.load(
    FOLD1_TRAIN_PATH,
    map_location="cpu",
    weights_only=False
)

validation_dataset = torch.load(
    FOLD1_VAL_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nTraining dataset")
print("Type:", type(train_dataset))
print("Samples:", len(train_dataset))

print("\nValidation dataset")
print("Type:", type(validation_dataset))
print("Samples:", len(validation_dataset))

print("\nTraining dataset parameters:")
print(train_dataset.get_parameters().keys())

print("\nValidation dataset parameters:")
print(validation_dataset.get_parameters().keys())

TFT configuration:
{
  "encoder_length": 56,
  "prediction_length": 28,
  "quantiles": [
    0.1,
    0.5,
    0.9
  ],
  "training_sampling": "weekly",
  "series_count": 200,
  "training_folds": {
    "development_fold_1": {
      "training_samples": 49200,
      "validation_samples": 200
    },
    "development_fold_2": {
      "training_samples": 50200,
      "validation_samples": 200
    },
    "development_fold_3": {
      "training_samples": 51200,
      "validation_samples": 200
    },
    "internal_test": {
      "training_samples": 52200,
      "validation_samples": 200
    },
    "official_evaluation": {
      "training_samples": 53200,
      "validation_samples": 200
    }
  }
}


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



Training dataset
Type: <class 'pytorch_forecasting.data.timeseries._timeseries.TimeSeriesDataSet'>
Samples: 49200

Validation dataset
Type: <class 'pytorch_forecasting.data.timeseries._timeseries.TimeSeriesDataSet'>
Samples: 200

Training dataset parameters:
dict_keys(['time_idx', 'target', 'group_ids', 'weight', 'max_encoder_length', 'min_encoder_length', 'min_prediction_idx', 'min_prediction_length', 'max_prediction_length', 'static_categoricals', 'static_reals', 'time_varying_known_categoricals', 'time_varying_known_reals', 'time_varying_unknown_categoricals', 'time_varying_unknown_reals', 'variable_groups', 'constant_fill_strategy', 'allow_missing_timesteps', 'lags', 'add_relative_time_idx', 'add_target_scales', 'add_encoder_length', 'target_normalizer', 'categorical_encoders', 'scalers', 'randomize_length', 'predict_mode'])

Validation dataset parameters:
dict_keys(['time_idx', 'target', 'group_ids', 'weight', 'max_encoder_length', 'min_encoder_length', 'min_prediction_idx', 'min

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
import os
import json
import gc
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import lightning as L
import pytorch_forecasting as pf

from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    LearningRateMonitor
)

SEED = 42

seed_everything(SEED, workers=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

TFT_INPUT_DIR = Path(
    "/kaggle/input/datasets/ryancjc/tft-salesf/tft"
)

OUTPUT_DIR = Path("/kaggle/working/tft")

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
RESULT_DIR = OUTPUT_DIR / "results"
CONFIG_DIR = OUTPUT_DIR / "configs"
PREDICTION_DIR = OUTPUT_DIR / "predictions"

for directory in [
    CHECKPOINT_DIR,
    RESULT_DIR,
    CONFIG_DIR,
    PREDICTION_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

with open(TFT_INPUT_DIR / "tft_dataset_config.json", "r") as f:
    TFT_CONFIG = json.load(f)

ENCODER_LENGTH = TFT_CONFIG["encoder_length"]
PREDICTION_LENGTH = TFT_CONFIG["prediction_length"]
QUANTILES = TFT_CONFIG["quantiles"]

FOLDS = [
    "development_fold_1",
    "development_fold_2",
    "development_fold_3"
]

EXPECTED_SAMPLES = {
    "development_fold_1": 49200,
    "development_fold_2": 50200,
    "development_fold_3": 51200
}

print("Configuration loaded")
print("Encoder length:", ENCODER_LENGTH)
print("Prediction length:", PREDICTION_LENGTH)
print("Quantiles:", QUANTILES)
print("Series count:", TFT_CONFIG["series_count"])
print("GPU:", torch.cuda.get_device_name(0))


Seed set to 42


Configuration loaded
Encoder length: 56
Prediction length: 28
Quantiles: [0.1, 0.5, 0.9]
Series count: 200
GPU: Tesla T4


In [ ]:
def load_tft_dataset(path):
    return torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )


TFT_DATASETS = {}

for fold in FOLDS:

    train_path = (
        TFT_INPUT_DIR / f"{fold}_training.pt"
    )

    val_path = (
        TFT_INPUT_DIR / f"{fold}_validation.pt"
    )

    train_dataset = load_tft_dataset(train_path)
    val_dataset = load_tft_dataset(val_path)

    assert len(train_dataset) == EXPECTED_SAMPLES[fold]
    assert len(val_dataset) == 200

    TFT_DATASETS[fold] = {
        "train": train_dataset,
        "validation": val_dataset
    }

    print(f"\n{fold}")
    print("Training samples:", len(train_dataset))
    print("Validation samples:", len(val_dataset))
    print("Encoder length:", train_dataset.max_encoder_length)
    print("Prediction length:", train_dataset.max_prediction_length)
    print("Target:", train_dataset.target)

print("\nAll development datasets passed verification.")

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



development_fold_1
Training samples: 49200
Validation samples: 200
Encoder length: 56
Prediction length: 28
Target: sales


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or 


development_fold_2
Training samples: 50200
Validation samples: 200
Encoder length: 56
Prediction length: 28
Target: sales


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



development_fold_3
Training samples: 51200
Validation samples: 200
Encoder length: 56
Prediction length: 28
Target: sales

All development datasets passed verification.


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
BASELINE_PARAMS = {
    "hidden_size": 32,
    "attention_head_size": 4,
    "lstm_layers": 1,
    "dropout": 0.10,
    "hidden_continuous_size": 16,
    "learning_rate": 0.001,
    "batch_size": 64
}

MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
NUM_WORKERS = 0


def create_dataloaders(
    train_dataset,
    val_dataset,
    batch_size=64
):

    train_loader = train_dataset.to_dataloader(
        train=True,
        batch_size=batch_size,
        num_workers=NUM_WORKERS
    )

    val_loader = val_dataset.to_dataloader(
        train=False,
        batch_size=batch_size,
        num_workers=NUM_WORKERS
    )

    return train_loader, val_loader


def create_tft_model(
    train_dataset,
    params
):

    model = TemporalFusionTransformer.from_dataset(
        train_dataset,

        hidden_size=params["hidden_size"],
        attention_head_size=params["attention_head_size"],
        lstm_layers=params["lstm_layers"],

        dropout=params["dropout"],
        hidden_continuous_size=params["hidden_continuous_size"],

        output_size=len(QUANTILES),
        loss=QuantileLoss(quantiles=QUANTILES),

        learning_rate=params["learning_rate"],
        reduce_on_plateau_patience=4,

        log_interval=-1,
        log_val_interval=-1,

        optimizer="adam"
    )

    return model


def create_trainer(
    max_epochs=MAX_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    checkpoint_callback=None,
    enable_checkpointing=True
):

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            min_delta=0.0001,
            patience=patience,
            mode="min",
            verbose=True
        )
    ]

    if checkpoint_callback is not None:
        callbacks.append(checkpoint_callback)

    trainer = Trainer(
        max_epochs=max_epochs,
        accelerator="gpu",
        devices=1,
        precision="32-true",
        gradient_clip_val=0.1,
        callbacks=callbacks,
        enable_checkpointing=enable_checkpointing,
        enable_model_summary=True,
        enable_progress_bar=True,
        logger=False,
        deterministic="warn",
        log_every_n_steps=50
    )

    return trainer

In [ ]:
fold = FOLDS[0]

train_dataset = TFT_DATASETS[fold]["train"]
val_dataset = TFT_DATASETS[fold]["validation"]

train_loader, val_loader = create_dataloaders(
    train_dataset,
    val_dataset,
    batch_size=BASELINE_PARAMS["batch_size"]
)

model = create_tft_model(
    train_dataset,
    BASELINE_PARAMS
)

batch = next(iter(train_loader))

x, y = batch

model.eval()

with torch.no_grad():
    output = model(x)

print("Forward pass successful")

print("Prediction shape:", output["prediction"].shape)

print("Expected prediction dimensions:",
      (BASELINE_PARAMS["batch_size"], PREDICTION_LENGTH, len(QUANTILES)))

print("Parameter count:",
      sum(p.numel() for p in model.parameters()))

assert output["prediction"].shape[1:] == (
    PREDICTION_LENGTH,
    len(QUANTILES)
)

del model, batch, x, y, output
gc.collect()
torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


Forward pass successful
Prediction shape: torch.Size([64, 28, 3])
Expected prediction dimensions: (64, 28, 3)
Parameter count: 134484


In [ ]:
BASELINE_RESULTS = []

for fold in FOLDS:

    print("\n" + "=" * 60)
    print("Training:", fold)
    print("=" * 60)

    seed_everything(SEED, workers=True)

    train_dataset = TFT_DATASETS[fold]["train"]
    val_dataset = TFT_DATASETS[fold]["validation"]

    train_loader, val_loader = create_dataloaders(
        train_dataset,
        val_dataset,
        batch_size=BASELINE_PARAMS["batch_size"]
    )

    model = create_tft_model(
        train_dataset,
        BASELINE_PARAMS
    )

    fold_checkpoint_dir = CHECKPOINT_DIR / "baseline"
    fold_checkpoint_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=fold_checkpoint_dir,
        filename=f"{fold}_baseline",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        save_last=False
    )

    trainer = create_trainer(
        max_epochs=MAX_EPOCHS,
        patience=EARLY_STOPPING_PATIENCE,
        checkpoint_callback=checkpoint_callback
    )

    start_time = time.time()

    trainer.fit(
        model,
        train_dataloaders=train_loader,
        val_dataloaders=val_loader
    )

    elapsed = (time.time() - start_time) / 60

    best_score = float(checkpoint_callback.best_model_score.cpu())
    best_epoch = checkpoint_callback.best_model_path

    result = {
        "fold": fold,
        "best_val_loss": best_score,
        "best_checkpoint": best_epoch,
        "training_time_minutes": elapsed,
        "best_epoch": trainer.current_epoch + 1
    }

    BASELINE_RESULTS.append(result)

    print("\nTraining completed")
    print("Best validation loss:", best_score)
    print("Checkpoint:", best_epoch)
    print("Training time:", round(elapsed, 2), "minutes")

    del model, trainer, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()


baseline_df = pd.DataFrame(BASELINE_RESULTS)

baseline_df.to_csv(
    RESULT_DIR / "baseline_training_summary.csv",
    index=False
)

print("\nBaseline training summary:")
display(baseline_df)

Seed set to 42



Training: development_fold_1


/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │  6.3 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    704 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  7.0 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 44.4 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 24.3 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  2.6 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     99 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 134 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 134 K                                                                                                
Total estimated model params size (MB): 0.538                                                                      
Modules in train mode: 697                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

... (2 outputs left out)

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: 
upsample_linear1d_backward_out_cuda does not have a deterministic implementation, but you set 
'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at 
https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. 
(Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass

Metric val_loss improved. New best score: 0.692
Metric val_loss improved by 0.015 >= min_delta = 0.0001. New best score: 0.677
Metric val_loss improved by 0.008 >= min_delta = 0.0001. New best score: 0.669


In [ ]:
def extract_prediction_and_target(prediction_result):

    predictions = prediction_result.output

    if isinstance(predictions, dict):
        predictions = predictions["prediction"]

    if isinstance(predictions, (tuple, list)):
        predictions = predictions[0]

    actual = prediction_result.y

    if isinstance(actual, (tuple, list)):
        actual = actual[0]

    predictions = predictions.detach().cpu().numpy()
    actual = actual.detach().cpu().numpy()

    return predictions, actual


def calculate_forecast_metrics(
    actual,
    predictions
):

    actual = np.asarray(actual).reshape(-1)
    predictions = np.asarray(predictions).reshape(-1, len(QUANTILES))

    p10 = predictions[:, 0]
    p50 = predictions[:, 1]
    p90 = predictions[:, 2]

    errors = actual - p50

    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))

    total_actual = np.sum(np.abs(actual))

    wape = (
        np.sum(np.abs(errors)) / total_actual * 100
        if total_actual > 0 else np.nan
    )

    result = {
        "MAE_P50": mae,
        "RMSE_P50": rmse,
        "WAPE_P50_pct": wape
    }

    for i, q in enumerate(QUANTILES):

        forecast = predictions[:, i]
        error = actual - forecast

        pinball = np.mean(
            np.maximum(q * error, (q - 1) * error)
        )

        result[f"Pinball_P{int(q * 100)}"] = pinball

    result["P10_coverage"] = np.mean(actual <= p10)
    result["P90_coverage"] = np.mean(actual <= p90)

    result["Interval_coverage"] = np.mean(
        (actual >= p10) & (actual <= p90)
    )

    result["Interval_width"] = np.mean(p90 - p10)

    result["Quantile_crossing_rate"] = np.mean(
        (p10 > p50) | (p50 > p90)
    )

    return result


def evaluate_tft_checkpoint(
    checkpoint_path,
    train_dataset,
    val_dataset,
    batch_size=64
):

    val_loader = val_dataset.to_dataloader(
        train=False,
        batch_size=batch_size,
        num_workers=NUM_WORKERS
    )

    model = TemporalFusionTransformer.load_from_checkpoint(
        checkpoint_path,
        map_location="cuda"
    )

    model.eval()

    prediction_result = model.predict(
        val_loader,
        mode="prediction",
        return_y=True,
        trainer_kwargs={
            "accelerator": "gpu",
            "devices": 1,
            "precision": "32-true",
            "logger": False,
            "enable_progress_bar": True
        }
    )

    predictions, actual = extract_prediction_and_target(
        prediction_result
    )

    assert predictions.shape == (
        len(val_dataset),
        PREDICTION_LENGTH,
        len(QUANTILES)
    )

    assert actual.shape == (
        len(val_dataset),
        PREDICTION_LENGTH
    )

    metrics = calculate_forecast_metrics(
        actual,
        predictions
    )

    del model, val_loader, prediction_result
    gc.collect()
    torch.cuda.empty_cache()

    return metrics, predictions, actual


BASELINE_EVALUATION = []
BASELINE_PREDICTIONS = {}

for fold in FOLDS:

    checkpoint_path = (
        CHECKPOINT_DIR / "baseline" / f"{fold}_baseline.ckpt"
    )

    if not checkpoint_path.exists():
        raise FileNotFoundError(checkpoint_path)

    metrics, predictions, actual = evaluate_tft_checkpoint(
        checkpoint_path,
        TFT_DATASETS[fold]["train"],
        TFT_DATASETS[fold]["validation"],
        batch_size=BASELINE_PARAMS["batch_size"]
    )

    metrics["fold"] = fold
    BASELINE_EVALUATION.append(metrics)

    BASELINE_PREDICTIONS[fold] = {
        "predictions": predictions,
        "actual": actual
    }

    np.savez_compressed(
        PREDICTION_DIR / f"{fold}_baseline_predictions.npz",
        predictions=predictions,
        actual=actual
    )

baseline_evaluation_df = pd.DataFrame(BASELINE_EVALUATION)

baseline_evaluation_df.to_csv(
    RESULT_DIR / "baseline_evaluation_summary.csv",
    index=False
)

display(baseline_evaluation_df)

### Fine-tuning

In [ ]:
import os
import gc
import json
import time
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import optuna

from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    Callback
)

from pytorch_forecasting import (
    TemporalFusionTransformer,
    TimeSeriesDataSet
)

from pytorch_forecasting.metrics import QuantileLoss

SEED = 42

seed_everything(SEED, workers=True)

TFT_INPUT_DIR = Path(
    "/kaggle/input/datasets/ryancjc/tft-salesf/tft"
)

TFT_OUTPUT_DIR = Path("/kaggle/working/tft")

OPTUNA_DIR = TFT_OUTPUT_DIR / "optuna"
RESULTS_DIR = TFT_OUTPUT_DIR / "results"
CHECKPOINT_DIR = OPTUNA_DIR / "temporary_checkpoints"

for directory in [
    OPTUNA_DIR,
    RESULTS_DIR,
    CHECKPOINT_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = TFT_INPUT_DIR / "tft_dataset_config.json"

with open(CONFIG_PATH, "r") as f:
    TFT_CONFIG = json.load(f)

ENCODER_LENGTH = TFT_CONFIG["encoder_length"]
PREDICTION_LENGTH = TFT_CONFIG["prediction_length"]
QUANTILES = TFT_CONFIG["quantiles"]

FOLDS = [
    "development_fold_1",
    "development_fold_2",
    "development_fold_3"
]

print("PyTorch:", torch.__version__)
print("Optuna:", optuna.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nTFT configuration:")
print("Encoder length:", ENCODER_LENGTH)
print("Prediction length:", PREDICTION_LENGTH)
print("Quantiles:", QUANTILES)

Seed set to 42


PyTorch: 2.11.0+cu130
Optuna: 5.0.0
GPU available: True
GPU: Tesla T4

TFT configuration:
Encoder length: 56
Prediction length: 28
Quantiles: [0.1, 0.5, 0.9]


In [ ]:
TFT_DATASETS = {}

for fold in FOLDS:

    print(f"\nLoading {fold}...", flush=True)

    train_path = TFT_INPUT_DIR / f"{fold}_training.pt"
    validation_path = TFT_INPUT_DIR / f"{fold}_validation.pt"

    assert train_path.exists(), f"Missing: {train_path}"
    assert validation_path.exists(), f"Missing: {validation_path}"

    train_dataset = torch.load(
        train_path,
        map_location="cpu",
        weights_only=False
    )

    validation_dataset = torch.load(
        validation_path,
        map_location="cpu",
        weights_only=False
    )

    assert isinstance(train_dataset, TimeSeriesDataSet)
    assert isinstance(validation_dataset, TimeSeriesDataSet)

    TFT_DATASETS[fold] = {
        "train": train_dataset,
        "validation": validation_dataset
    }

    print(
        f"Training samples: {len(train_dataset):,} | "
        f"Validation samples: {len(validation_dataset):,}",
        flush=True
    )

print("\nAll development datasets loaded.")


Loading development_fold_1...


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Training samples: 49,200 | Validation samples: 200

Loading development_fold_2...


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Training samples: 50,200 | Validation samples: 200

Loading development_fold_3...


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Training samples: 51,200 | Validation samples: 200

All development datasets loaded.


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
def create_tft_model(training_dataset, params):

    model = TemporalFusionTransformer.from_dataset(
        training_dataset,

        learning_rate=params["learning_rate"],

        hidden_size=params["hidden_size"],
        attention_head_size=params["attention_head_size"],
        lstm_layers=params["lstm_layers"],
        dropout=params["dropout"],
        hidden_continuous_size=params["hidden_continuous_size"],

        output_size=len(QUANTILES),

        loss=QuantileLoss(
            quantiles=QUANTILES
        ),

        log_interval=-1,
        reduce_on_plateau_patience=4
    )

    return model

In [ ]:
class OptunaProgressCallback(Callback):

    def on_validation_end(self, trainer, pl_module):

        if trainer.sanity_checking:
            return

        val_loss = trainer.callback_metrics.get("val_loss")

        if val_loss is not None:

            print(
                f"Epoch {trainer.current_epoch + 1} | "
                f"Validation loss: {float(val_loss):.6f}",
                flush=True
            )

In [ ]:
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
NUM_WORKERS = 3

def create_optuna_trainer(
    trial_number,
    fold_number,
    patience=EARLY_STOPPING_PATIENCE
):

    early_stopping = EarlyStopping(
        monitor="val_loss",
        min_delta=0.0001,
        patience=patience,
        mode="min",
        verbose=True
    )

    checkpoint_dir = (
        CHECKPOINT_DIR
        / f"trial_{trial_number}_fold_{fold_number}"
    )

    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    checkpoint_callback = ModelCheckpoint(
        dirpath=checkpoint_dir,
        filename="best",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        save_last=False
    )

    trainer = Trainer(
        max_epochs=MAX_EPOCHS,

        accelerator="gpu",
        devices=1,
        precision="32-true",

        gradient_clip_val=0.1,

        callbacks=[
            early_stopping,
            checkpoint_callback,
            OptunaProgressCallback()
        ],

        enable_checkpointing=True,
        enable_model_summary=False,
        enable_progress_bar=False,

        logger=False,

        deterministic="warn",
        num_sanity_val_steps=0,
        log_every_n_steps=50
    )

    return trainer, early_stopping, checkpoint_callback

In [ ]:
def suggest_tft_parameters(trial):

    hidden_size = trial.suggest_categorical(
        "hidden_size",
        [16, 32, 64]
    )

    attention_head_size = trial.suggest_categorical(
        "attention_head_size",
        [1, 2, 4]
    )

    hidden_continuous_candidates = [
        value
        for value in [8, 16, 32]
        if value <= hidden_size
    ]

    hidden_continuous_size = trial.suggest_categorical(
        "hidden_continuous_size",
        hidden_continuous_candidates
    )

    params = {
        "hidden_size": hidden_size,

        "attention_head_size": attention_head_size,

        "lstm_layers": trial.suggest_categorical(
            "lstm_layers",
            [1, 2]
        ),

        "dropout": trial.suggest_float(
            "dropout",
            0.05,
            0.25
        ),

        "hidden_continuous_size": hidden_continuous_size,

        "learning_rate": trial.suggest_float(
            "learning_rate",
            1e-4,
            3e-3,
            log=True
        ),

        "batch_size": trial.suggest_categorical(
            "batch_size",
            [32, 64]
        )
    }

    return params

In [ ]:
def objective(trial):

    params = suggest_tft_parameters(trial)

    print("\n" + "=" * 70, flush=True)
    print(f"STARTING OPTUNA TRIAL {trial.number}", flush=True)
    print("Parameters:", params, flush=True)
    print("=" * 70, flush=True)

    fold_losses = []

    for fold_index, fold in enumerate(FOLDS, start=1):

        print(
            f"\nTrial {trial.number} | "
            f"Training {fold} ({fold_index}/3)",
            flush=True
        )

        train_dataset = TFT_DATASETS[fold]["train"]
        validation_dataset = TFT_DATASETS[fold]["validation"]

        train_loader = train_dataset.to_dataloader(
            train=True,
            batch_size=params["batch_size"],
            num_workers=NUM_WORKERS
        )

        validation_loader = validation_dataset.to_dataloader(
            train=False,
            batch_size=params["batch_size"],
            num_workers=NUM_WORKERS
        )

        model = create_tft_model(
            train_dataset,
            params
        )

        trainer, early_stopping, checkpoint_callback = (
            create_optuna_trainer(
                trial.number,
                fold_index
            )
        )

        try:

            start_time = time.time()

            trainer.fit(
                model,
                train_dataloaders=train_loader,
                val_dataloaders=validation_loader
            )

            training_time = (
                time.time() - start_time
            ) / 60

            best_score = checkpoint_callback.best_model_score

            if best_score is None:
                raise RuntimeError(
                    f"No best validation score recorded for {fold}"
                )

            best_val_loss = float(
                best_score.detach().cpu()
            )

            best_checkpoint_path = (
                checkpoint_callback.best_model_path
            )

            if not best_checkpoint_path:
                raise RuntimeError(
                    f"Best checkpoint path missing for {fold}"
                )

            best_checkpoint = torch.load(
                best_checkpoint_path,
                map_location="cpu",
                weights_only=False
            )

            best_epoch = int(best_checkpoint["epoch"]) + 1

            fold_losses.append(best_val_loss)

            print(
                f"\nCompleted {fold}",
                flush=True
            )

            print(
                f"Best validation loss: {best_val_loss:.6f}",
                flush=True
            )

            print(
                f"Best epoch: {best_epoch}",
                flush=True
            )

            print(
                f"Training time: {training_time:.2f} minutes",
                flush=True
            )

            trial.set_user_attr(
                f"{fold}_best_val_loss",
                best_val_loss
            )

            trial.set_user_attr(
                f"{fold}_best_epoch",
                best_epoch
            )

            trial.set_user_attr(
                f"{fold}_training_time_minutes",
                training_time
            )

        finally:

            del model
            del trainer
            del train_loader
            del validation_loader
            del early_stopping
            del checkpoint_callback

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            shutil.rmtree(
                CHECKPOINT_DIR / f"trial_{trial.number}_fold_{fold_index}",
                ignore_errors=True
            )

    mean_validation_loss = float(np.mean(fold_losses))

    trial.set_user_attr(
        "fold_validation_losses",
        fold_losses
    )

    trial.set_user_attr(
        "mean_validation_loss",
        mean_validation_loss
    )

    print("\n" + "=" * 70, flush=True)
    print(f"TRIAL {trial.number} COMPLETED", flush=True)
    print("Fold losses:", fold_losses, flush=True)
    print(
        f"Mean validation loss: {mean_validation_loss:.6f}",
        flush=True
    )
    print("=" * 70, flush=True)

    return mean_validation_loss

In [ ]:
OPTUNA_DB_PATH = OPTUNA_DIR / "tft_optuna_study.db"

STUDY_NAME = "tft_development_hyperparameter_tuning"

storage = f"sqlite:///{OPTUNA_DB_PATH}"

sampler = optuna.samplers.TPESampler(
    seed=SEED,
    n_startup_trials=1
)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=storage,
    direction="minimize",
    sampler=sampler,
    pruner=optuna.pruners.NopPruner(),
    load_if_exists=True
)

TARGET_TRIALS = 3

completed_trial_count = sum(
    trial.state == optuna.trial.TrialState.COMPLETE
    for trial in study.trials
)

remaining_trials = max(
    0,
    TARGET_TRIALS - completed_trial_count
)

print("Existing trials:", len(study.trials))
print("Completed trials:", completed_trial_count)
print("Additional trials:", remaining_trials)

def save_optuna_progress(study, trial):

    trials_df = study.trials_dataframe(
        attrs=(
            "number",
            "value",
            "params",
            "state",
            "user_attrs"
        )
    )

    trials_df.to_csv(
        RESULTS_DIR / "tft_optuna_trial_results.csv",
        index=False
    )

    completed = [
        item for item in study.trials
        if item.state == optuna.trial.TrialState.COMPLETE
    ]

    if completed:

        best_trial = study.best_trial

        progress_config = {
            "study_name": STUDY_NAME,
            "completed_trials": len(completed),
            "best_trial_number": best_trial.number,
            "best_validation_loss": best_trial.value,
            "best_parameters": best_trial.params
        }

        with open(
            RESULTS_DIR / "tft_optuna_progress.json",
            "w"
        ) as f:
            json.dump(progress_config, f, indent=4)

    print(
        f"\nSaved Optuna progress after trial {trial.number}.",
        flush=True
    )


if remaining_trials > 0:

    study.optimize(
        objective,
        n_trials=remaining_trials,
        callbacks=[save_optuna_progress],
        gc_after_trial=True,
        show_progress_bar=False
    )

else:

    print("Target number of completed trials already reached.")

[I 2026-11-05 16:15:26,342] A new study created in RDB with name: tft_development_hyperparameter_tuning


Existing trials: 0
Completed trials: 0
Additional trials: 3

STARTING OPTUNA TRIAL 0
Parameters: {'hidden_size': 32, 'attention_head_size': 1, 'lstm_layers': 1, 'dropout': 0.2439819704323989, 'hidden_continuous_size': 16, 'learning_rate': 0.0016967533607196547, 'batch_size': 32}

Trial 0 | Training development_fold_1 (1/3)


/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, L

Epoch 1 | Validation loss: 0.708648


Metric val_loss improved. New best score: 0.709


Epoch 2 | Validation loss: 0.685492


... (140 outputs left out)

Monitored metric val_loss did not improve in the last 5 records. Best score: 0.655. Signaling Trainer to stop.



Completed development_fold_3
Best validation loss: 0.655102
Best epoch: 13
Training time: 83.56 minutes

TRIAL 2 COMPLETED
Fold losses: [0.661380410194397, 0.7158172726631165, 0.655102014541626]
Mean validation loss: 0.677433


[I 2026-11-06 02:43:44,494] Trial 2 finished with value: 0.6774332324663798 and parameters: {'hidden_size': 32, 'attention_head_size': 2, 'hidden_continuous_size': 16, 'lstm_layers': 1, 'dropout': 0.23719641420728668, 'learning_rate': 0.000835308773021013, 'batch_size': 32}. Best is trial 2 with value: 0.6774332324663798.



Saved Optuna progress after trial 2.


In [ ]:
if not any(
    trial.state == optuna.trial.TrialState.COMPLETE
    for trial in study.trials
):
    raise RuntimeError("No completed Optuna trials available.")

best_trial = study.best_trial

BEST_TFT_PARAMS = {
    "hidden_size": best_trial.params["hidden_size"],
    "attention_head_size": best_trial.params["attention_head_size"],
    "lstm_layers": best_trial.params["lstm_layers"],
    "dropout": best_trial.params["dropout"],
    "hidden_continuous_size": best_trial.params["hidden_continuous_size"],
    "learning_rate": best_trial.params["learning_rate"],
    "batch_size": best_trial.params["batch_size"]
}

BEST_TFT_RESULT = {
    "model_name": "Temporal Fusion Transformer",
    "study_name": STUDY_NAME,
    "best_trial_number": best_trial.number,
    "best_mean_validation_loss": best_trial.value,
    "best_hyperparameters": BEST_TFT_PARAMS,
    "fold_validation_losses": best_trial.user_attrs.get(
        "fold_validation_losses"
    ),
    "fold_best_epochs": {
        fold: best_trial.user_attrs.get(f"{fold}_best_epoch")
        for fold in FOLDS
    },
    "fold_training_times_minutes": {
        fold: best_trial.user_attrs.get(
            f"{fold}_training_time_minutes"
        )
        for fold in FOLDS
    },
    "completed_trials": sum(
        trial.state == optuna.trial.TrialState.COMPLETE
        for trial in study.trials
    ),
    "random_seed": SEED
}

BEST_PARAMS_PATH = (
    RESULTS_DIR / "tft_best_hyperparameters.json"
)

with open(BEST_PARAMS_PATH, "w") as f:
    json.dump(BEST_TFT_RESULT, f, indent=4)

TRIAL_RESULTS_PATH = (
    RESULTS_DIR / "tft_optuna_trial_results.csv"
)

study.trials_dataframe(
    attrs=(
        "number",
        "value",
        "params",
        "state",
        "user_attrs"
    )
).to_csv(
    TRIAL_RESULTS_PATH,
    index=False
)

print("Best trial:", best_trial.number)
print("Best mean validation loss:", best_trial.value)

print("\nBest hyperparameters:")

for key, value in BEST_TFT_PARAMS.items():
    print(f"{key}: {value}")

print("\nSaved:")
print(BEST_PARAMS_PATH)
print(TRIAL_RESULTS_PATH)

Best trial: 2
Best mean validation loss: 0.6774332324663798

Best hyperparameters:
hidden_size: 32
attention_head_size: 2
lstm_layers: 1
dropout: 0.23719641420728668
hidden_continuous_size: 16
learning_rate: 0.000835308773021013
batch_size: 32

Saved:
/kaggle/working/tft/results/tft_best_hyperparameters.json
/kaggle/working/tft/results/tft_optuna_trial_results.csv


In [ ]:
BASELINE_MEAN_VAL_LOSS = 0.674543

TUNED_MEAN_VAL_LOSS = study.best_value

improvement = (
    (BASELINE_MEAN_VAL_LOSS - TUNED_MEAN_VAL_LOSS)
    / BASELINE_MEAN_VAL_LOSS
) * 100

print("Baseline mean validation loss:",
      round(BASELINE_MEAN_VAL_LOSS, 6))

print("Tuned mean validation loss:",
      round(TUNED_MEAN_VAL_LOSS, 6))

print("Relative improvement:",
      round(improvement, 4), "%")

if improvement > 0:
    print("Tuned TFT has lower validation loss than baseline.")

elif improvement < 0:
    print("Tuned TFT has higher validation loss than baseline.")

else:
    print("Tuned TFT matches baseline validation loss.")

Baseline mean validation loss: 0.674543
Tuned mean validation loss: 0.677433
Relative improvement: -0.4285 %
Tuned TFT has higher validation loss than baseline.


In [ ]:
print("Optuna output files:")

for path in sorted(TFT_OUTPUT_DIR.rglob("*")):

    if path.is_file():

        print(
            path.relative_to(TFT_OUTPUT_DIR),
            f"| {path.stat().st_size / (1024 ** 2):.3f} MB"
        )

required_files = [
    OPTUNA_DB_PATH,
    BEST_PARAMS_PATH,
    TRIAL_RESULTS_PATH
]

for path in required_files:
    assert path.exists(), f"Missing artifact: {path}"

print("\nOPTUNA ARTIFACT VERIFICATION: PASSED")

Optuna output files:
optuna/tft_optuna_study.db | 0.109 MB
results/tft_best_hyperparameters.json | 0.001 MB
results/tft_optuna_progress.json | 0.000 MB
results/tft_optuna_trial_results.csv | 0.001 MB

OPTUNA ARTIFACT VERIFICATION: PASSED
